<a href="https://colab.research.google.com/github/solivagvs/stat554hw05/blob/main/sandorHW05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Student: Bryan Sandor**

**Stat 554**

**Homework 05**

# Part I - Concepts

## Question 1

Like many individuals, I concern myself with the weather. The National Oceanic and Atmospheric Administration (NOAA), among other entities, collects a surfeit of data to make decisions regarding forecasts to one of its appendant bodies, the National Weather Service (NWS).
- Volume: The agency must collect a variety of data per sensor measuring things such as barometric pressure, temperature, humidity, light, air quality, etc.
- Velocity: These conditions change by the second and the NWS relies on up-to-date information in order to make more accurate predictions.
- Variety: Many subsidiary agencies exist to furnish the NWS with supplementary data, including that they collect theirself.
- Veracity: The more accurate the data collected, presumably the better a forecast the NWS can make.
- Value: Data collected and predictions made help inform future forecasts and increase reliability.

## Question 2

- **C**reate: After connecting to the database, we create a local data frame instance to peruse the data. Furthermore, we create a new column using the existing data, in the last step, to return the win/loss percentages.
- **R**ead: We read the data several different times, subject to different specified conditions.
- **U**pdate: I don't see a case where the database is updated apart from creating a win/loss variable and "updating" it using existing data.
- **D**elete: By closing the connection to the database, the local data frame is purged from memory upon disconnecting from the Google Colab server.

## Question 3

When using the `SELECT` statement, data may be grouped using the `GROUP BY` command specifically subject to the optional `HAVING` clause.

# Part II - Simulation of a Sampling Distribution

## Question 4

We first use the supplied code to generate random data from a simple linear regression (SLR) model $$Y_i = \beta_0 + \beta_1 x_i + E_i$$ where the $E_i$ are indepedent and identically distributed (iid) from $N(0, \sigma^2 = 1)$. This data is generated by assuming values for $\beta_0$, $\beta_1$, $n$, and a sequence of $x$ values:

In [ ]:
#import some modules needed
import matplotlib.pyplot as plt
import numpy as np
from numpy.random import default_rng

rng = default_rng(999)
beta_0 = 7
beta_1 = 1.5

# get three 'values' of x at each integer from 0 to 10.
x = np.array(list(np.linspace(start = 0, stop = 10, num = 11))*3)
n = 33

#create the 'responses' modeled from the line plus a random deviation
y = beta_0 + beta_1 * x + rng.standard_normal(n)

#visualize the data
plt.scatter(x = x, y = y)
plt.show()

Now we perform regression on the 33 generated values to find an estimate for the slope of the SLR, $\hat{\beta_1}$.

In [ ]:
from sklearn.linear_model import LinearRegression

x = x.reshape((-1, 1))
reg = LinearRegression().fit(x, y)

LinearRegression()
m = reg.coef_[0]
print(m)

Now we create use a loop to repeat this process and generate 5000 different estimates for the slope.

In [ ]:
k = 5000

beta1hat = [0] * k

for i in range(k):
    # get three 'values' of x at each integer from 0 to 10.
    x = np.array(list(np.linspace(start = 0, stop = 10, num = 11))*3)
    n = 33

    #create the 'responses' modeled from the line plus a random deviation
    y = beta_0 + beta_1 * x + rng.standard_normal(n)

    x = x.reshape((-1, 1))
    reg = LinearRegression().fit(x, y)

    LinearRegression()
    beta1hat[i] = reg.coef_[0]

Now we generate a histogram of the different estimates for $\beta_1$ to examine their distribution.

In [ ]:
import seaborn as sns

sns.histplot(beta1hat)

Finally, we use the generated values to find the empirical probability $$\mathbb{P}(\hat{\beta_1} > 1.65)$$

In [ ]:
probcount = 0

for i in range(k):
    if beta1hat[i] > 1.65:
        probcount += 1

print("The probability beta-1-hat is greater than 1.65 is", \
      round(probcount / k, 4))

This probability is much lower than, say, 0.05, so we can say it is statistically significant. In a hypothesis test, we could conclude a value for $\beta_1$ greater than 1.65 is "unusual."

# Part III - Big Data Examples & Rare Events

## Question 5

## Question 6

## Question 7

The article found at https://allendowney.substack.com/p/superbolts?utm_source=substack&utm_medium=email addresses a "rare" (compared by frequency) phenomenon involving superbolts of lightning. While "rare" in comparison to the frequency of typical lightning strikes, lightning strikes themselves occur with enough frequency that these superbolts can be observed regularly.

I liken this to the incidence of cancer in people. The event is "rare" enough, but there is a sufficient quantity of people on earth so that even a rare event occurs with enough regularity to seek prediction models.

# Part IV - Querying a database

For this part, we will use an external database (stored locally in Colab) containing information regarding baseball.

## Question 8

First we read in the data and store it as a data frame.

In [ ]:
import pandas as pd
import sqlite3


# in-memory (local) database
con = sqlite3.connect("lahman_1871-2022.sqlite")

#create a 'cursor' object from our connection
cursor = con.cursor()

#SQL query to return all table names in the data base
#The * indicates we want to select everything
get_schema = '''
        SELECT *
        FROM sqlite_schema
        WHERE type = "table";
        '''

#execute the SQL query on the database!
cursor.execute(get_schema)

# read the table in as a dataframe and view it
baseball = pd.read_sql(get_schema, con)
baseball

## Question 9

Next  we return all the teams that played in the year 2015 with all the corresponding columns from the `Teams` table. Note the appropriate year column name is given from the `sql` column in the `Teams` row as `yearID`.

In [ ]:
q9query = '''
        SELECT *
        FROM "Teams"
        WHERE yearID = "2015";
        '''

#execute the SQL query on the database!
cursor.execute(q9query)

# read the table in as a dataframe and view it
teams2015 = pd.read_sql(q9query, con)
teams2015

## Question 10

Now we return all the players in the hall of fame, the year they were voted into the hall of fame, and their category.

In [ ]:
q10query = '''
        SELECT playerID, yearid, category
        FROM "HallOfFame"
        WHERE inducted = "Y"
        '''

#execute the SQL query on the database!
cursor.execute(q10query)

# read the table in as a dataframe and view it
hallOfFame = pd.read_sql(q10query, con)
hallOfFame

## Question 11

Next we return all unique managers of the Pittsburg Pirates and only that information from the `Managers` table.

In [ ]:
q11query = '''
        SELECT DISTINCT playerID
        FROM "Managers"
        WHERE teamID = "PIT"
        '''

#execute the SQL query on the database!
cursor.execute(q11query)

# read the table in as a dataframe and view it
piratesManagers = pd.read_sql(q11query, con)
piratesManagers

Now we use the `HallOfFame` and `Managers` tables to return all of the `playerID`s for the people that managed for a team that were also inducted into the hall of fame.

## Question 12

In [ ]:
q12query1 = """
  SELECT DISTINCT hall.playerID
  FROM "HallOfFame" AS hall
  INNER JOIN Managers AS man
  ON hall.playerID = man.playerID
  """

# read the table in as a dataframe and view it
hallOfFameManagers = pd.read_sql(q12query1, con)
hallOfFameManagers

We indicate the number of such players:

In [ ]:
q12query2 = """
  SELECT COUNT( DISTINCT hall.playerID )
  FROM "HallOfFame" AS hall
  INNER JOIN Managers AS man
  ON hall.playerID = man.playerID
  """

pd.read_sql(q12query2, con)

## Question 13

Next, we will use the same two tables, `HallOfFame` and `Managers`, to return every season managed by each manager that made it to the hall of fame.

In [ ]:
q13query1 = """
  SELECT DISTINCT hall.playerID, man.yearID, man.G, man.W, man.L
  FROM "HallOfFame" AS hall
  INNER JOIN Managers AS man
  ON hall.playerID = man.playerID
  """

# read the table in as a dataframe and view it
hallOfFameManagersFull = pd.read_sql(q13query1, con)
hallOfFameManagersFull

Next, we determine the overall win/loss records for each of these hall of fame managers.

In [ ]:
# sum wins and losses per player and group by the player

q13query2 = """
  SELECT DISTINCT hall.playerID, SUM(man.W), SUM(man.L)
  FROM "HallOfFame" AS hall
  INNER JOIN Managers AS man
  ON hall.playerID = man.playerID
  GROUP BY hall.playerID
  """

# read the table in as a dataframe and view it
hallOfFameManagersRecord = pd.read_sql(q13query2, con)
hallOfFameManagersRecord

Then we create a new variable that is the win/loss percentage.

In [ ]:
# must multiply the W column by 1.0 to change from integer division to float
# division

q13query3 = """
  SELECT DISTINCT hall.playerID, (man.W * 1.0 / man.G) AS "W/G"
  FROM "HallOfFame" AS hall
  INNER JOIN Managers AS man
  ON hall.playerID = man.playerID
  GROUP BY man.playerID
  """

# read the table in as a dataframe and view it
hallOfFameManagersProp = pd.read_sql(q13query3, con)
hallOfFameManagersProp

Finally, we sort the resulting data from largest to smallest and conclude by closing the cursor.

In [ ]:
# must multiply the W column by 1.0 to change from integer division to float
# division

q13query3 = """
  SELECT DISTINCT hall.playerID, (man.W * 1.0 / man.G) AS "W/G"
  FROM "HallOfFame" AS hall
  INNER JOIN Managers AS man
  ON hall.playerID = man.playerID
  GROUP BY man.playerID
  ORDER BY "W/G" DESC
  """

# read the table in as a dataframe and view it
hallOfFameManagersProp = pd.read_sql(q13query3, con)
hallOfFameManagersProp

In [ ]:
# close the cursor
cursor.close()